# California Housing Dataset

Implementación con PyTorch.

In [292]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.optim as optim


from torch import nn
from torch.utils.data import random_split, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset

In [293]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

Disclaimer: Hubo uso de IA como apoyo para corregir lógica en la que se manejaban los datos (Notablemente: Corregir la calculación de los `means` y `stds` en base a datos originales, en vez de calcularlos después del `Trimming` en la función `compute_preprocessing_params`).

# Obteniendo el dataset y partición

In [294]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

# Clases y funciones usadas

In [295]:
def compute_preprocessing_params(X_train, feature_mask):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    mask = feature_mask
    X_selected = X_train_tensor[:, mask]

    # Bounds
    Q1 = torch.quantile(X_selected, 0.25, dim=0)
    Q3 = torch.quantile(X_selected, 0.75, dim=0)
    IQR = Q3 - Q1

    lower_bounds = Q1 - 1.5 * IQR
    upper_bounds = Q3 + 1.5 * IQR

    # Trimming
    X_trimmed = X_selected.clone()
    X_trimmed = torch.clamp(X_trimmed, lower_bounds, upper_bounds)

    # means y stds después de Trimming (duh!!)
    means = X_trimmed.mean(dim=0)
    stds = X_trimmed.std(dim=0)

    return lower_bounds, upper_bounds, means, stds

In [296]:
class ScalingLayer(nn.Module):

    def __init__(self, mean: torch.Tensor, std: torch.Tensor, feature_mask: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        self.register_buffer('feature_mask', feature_mask)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Para no modificar el original
        X_scaled = X.clone()

        mask = self.feature_mask

        X_scaled[:, mask] = (X_scaled[:, mask] - self.mean) / (self.std + 1e-8)

        return X_scaled

In [297]:
class OutlierTrimmingLayer(nn.Module):

    def __init__(self, lower_bounds: torch.Tensor, upper_bounds: torch.Tensor, feature_mask: torch.Tensor):
        super().__init__()
        self.register_buffer('lower_bounds', lower_bounds)
        self.register_buffer('upper_bounds', upper_bounds)
        self.register_buffer('feature_mask', feature_mask)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Para no modificar el original
        X_trimmed = X.clone()

        mask = self.feature_mask

        X_trimmed[:, mask] = torch.clamp(
            X_trimmed[:, mask],
            self.lower_bounds,
            self.upper_bounds
        )

        return X_trimmed

In [298]:
class CaliforniaHousingNN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size, lower_bound, upper_bound, mean, std, feature_mask):
        super().__init__()

        # Preprocessing
        self.trim_layer = OutlierTrimmingLayer(
            lower_bound,
            upper_bound,
            feature_mask
        )
        self.scale_layer = ScalingLayer(
            mean,
            std,
            feature_mask
        )

        self.hidden = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        # Preprocessing
        x = self.trim_layer(x)
        x = self.scale_layer(x)

        # Main stuff
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [299]:
# Clase con otro hidden layer

class CaliforniaHousingNN_ALT(nn.Module):

    def __init__(self, input_size, hidden_size, hidden_size_dos, output_size, lower_bound, upper_bound, mean, std, feature_mask):
        super().__init__()

        self.trim_layer = OutlierTrimmingLayer(
            lower_bound,
            upper_bound,
            feature_mask
        )
        self.scale_layer = ScalingLayer(
            mean,
            std,
            feature_mask
        )

        self.hidden = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.hidden2 = nn.Linear(hidden_size, hidden_size_dos)
        self.output = nn.Linear(hidden_size_dos, output_size)

    def forward(self, x):

        x = self.trim_layer(x)
        x = self.scale_layer(x)

        x = self.hidden(x)
        x = self.relu(x)
        x = self.hidden2(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [300]:
def train(model, num_epochs, optimizer, batch_size):

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    loss_fn = nn.MSELoss()

    # Train
    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []

        for X_i, y_i in val_loader:
                y_pred = model(X_i)
                val_loss += loss_fn(y_pred, y_i).item()
                all_preds.append(y_pred)
                all_targets.append(y_i)

        y_pred_all = torch.cat(all_preds).flatten()
        y_true_all = torch.cat(all_targets).flatten()

        # R^2
        ss_res = torch.sum((y_true_all - y_pred_all) ** 2)
        ss_tot = torch.sum((y_true_all - torch.mean(y_true_all)) ** 2)
        r2 = 1 - ss_res / (ss_tot + 1e-8)

        # RMSE
        rmse = torch.sqrt(torch.mean((y_true_all - y_pred_all) ** 2))

        print(f"Epoch: {epoch+1} \tLoss: {val_loss/len(val_loader):.4f} \tR²: {r2:.4f} \tRMSE: ${rmse.item() * 100000:.2f}")

# Proceso principal (carga de datos)

In [301]:
# Convertir a tensores
X_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
X_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
X_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# Masking para ignorar Latitude y Longitude
features_to_process = [0, 1, 2, 3, 4, 5]
feature_mask = torch.tensor([True if i in features_to_process else False for i in range(8)])

lower_bounds, upper_bounds, means, stds = compute_preprocessing_params(x_train.values, feature_mask)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Red Neuronal 1

- Capa oculta: 100
- Épocas: 20
- Batches: 64
- Optimizer: Adam
- Learning Rate: 0.01

In [302]:
# Train model 1
model_m1 = CaliforniaHousingNN(8, 100, 1, lower_bounds, upper_bounds, means, stds, feature_mask)
optimizer_m1 = optim.Adam(model_m1.parameters(), lr=0.01)
batch_size_m1 = 64
epochs_m1 = 25
train(model_m1, epochs_m1, optimizer_m1, batch_size_m1)

Epoch: 1 	Loss: 0.5386 	R²: 0.5934 	RMSE: $72793.85
Epoch: 2 	Loss: 0.5614 	R²: 0.5741 	RMSE: $74505.08
Epoch: 3 	Loss: 0.6228 	R²: 0.5268 	RMSE: $78532.28
Epoch: 4 	Loss: 0.4783 	R²: 0.6406 	RMSE: $68440.43
Epoch: 5 	Loss: 0.5107 	R²: 0.6165 	RMSE: $70699.21
Epoch: 6 	Loss: 0.4757 	R²: 0.6431 	RMSE: $68206.25
Epoch: 7 	Loss: 0.7069 	R²: 0.4629 	RMSE: $83669.16
Epoch: 8 	Loss: 0.5239 	R²: 0.6040 	RMSE: $71839.19
Epoch: 9 	Loss: 0.4751 	R²: 0.6428 	RMSE: $68229.66
Epoch: 10 	Loss: 0.4478 	R²: 0.6630 	RMSE: $66273.05
Epoch: 11 	Loss: 0.4483 	R²: 0.6629 	RMSE: $66284.13
Epoch: 12 	Loss: 0.4399 	R²: 0.6693 	RMSE: $65653.30
Epoch: 13 	Loss: 0.4417 	R²: 0.6677 	RMSE: $65810.59
Epoch: 14 	Loss: 0.4432 	R²: 0.6664 	RMSE: $65942.73
Epoch: 15 	Loss: 0.4385 	R²: 0.6706 	RMSE: $65519.93
Epoch: 16 	Loss: 0.4657 	R²: 0.6507 	RMSE: $67472.61
Epoch: 17 	Loss: 0.4364 	R²: 0.6711 	RMSE: $65478.30
Epoch: 18 	Loss: 0.4407 	R²: 0.6675 	RMSE: $65828.22
Epoch: 19 	Loss: 0.4971 	R²: 0.6273 	RMSE: $69701.42
Ep

# Red Neuronal 2

- Capa oculta 1: 100
- Capa oculta 2: 125
- Épocas: 20
- Batches: 40
- Optimizer: Adam
- Learning Rate: 0.001

In [303]:
# Train model 2
model_m2 = CaliforniaHousingNN_ALT(8, 100, 125, 1, lower_bounds, upper_bounds, means, stds, feature_mask)
optimizer_m2 = optim.Adam(model_m2.parameters(), lr=0.001)
batch_size_m2 = 40
epochs_m2 = 20
train(model_m2, epochs_m2, optimizer_m2, batch_size_m2)

Epoch: 1 	Loss: 0.5553 	R²: 0.5750 	RMSE: $74422.25
Epoch: 2 	Loss: 0.6047 	R²: 0.5374 	RMSE: $77653.12
Epoch: 3 	Loss: 0.4608 	R²: 0.6476 	RMSE: $67768.47
Epoch: 4 	Loss: 0.4601 	R²: 0.6481 	RMSE: $67724.66
Epoch: 5 	Loss: 0.5686 	R²: 0.5648 	RMSE: $75310.51
Epoch: 6 	Loss: 0.5529 	R²: 0.5769 	RMSE: $74256.74
Epoch: 7 	Loss: 0.4440 	R²: 0.6605 	RMSE: $66517.25
Epoch: 8 	Loss: 0.4407 	R²: 0.6630 	RMSE: $66278.43
Epoch: 9 	Loss: 0.4837 	R²: 0.6301 	RMSE: $69438.86
Epoch: 10 	Loss: 0.4359 	R²: 0.6669 	RMSE: $65895.10
Epoch: 11 	Loss: 0.4723 	R²: 0.6388 	RMSE: $68612.96
Epoch: 12 	Loss: 0.5052 	R²: 0.6136 	RMSE: $70964.57
Epoch: 13 	Loss: 0.4721 	R²: 0.6391 	RMSE: $68580.71
Epoch: 14 	Loss: 0.5408 	R²: 0.5861 	RMSE: $73445.97
Epoch: 15 	Loss: 0.4380 	R²: 0.6650 	RMSE: $66076.78
Epoch: 16 	Loss: 0.5251 	R²: 0.5983 	RMSE: $72360.95
Epoch: 17 	Loss: 0.4422 	R²: 0.6619 	RMSE: $66386.33
Epoch: 18 	Loss: 0.4382 	R²: 0.6650 	RMSE: $66080.75
Epoch: 19 	Loss: 0.4243 	R²: 0.6755 	RMSE: $65033.75
Ep

# Red Neuronal 3

- Capa oculta: 100
- Épocas: 15
- Batches: 50
- Optimizer: SGD
- Learning Rate: 0.001

In [304]:
# Train model 3
model_m3 = CaliforniaHousingNN(8, 100, 1, lower_bounds, upper_bounds, means, stds, feature_mask)
optimizer_m3 = optim.SGD(model_m3.parameters(), lr=0.001)
batch_size_m3 = 50
epochs_m3 = 15
train(model_m3, epochs_m3, optimizer_m3, batch_size_m3)

Epoch: 1 	Loss: 1.9629 	R²: -0.4943 	RMSE: $139558.43
Epoch: 2 	Loss: 1.4844 	R²: -0.1316 	RMSE: $121446.47
Epoch: 3 	Loss: 1.3560 	R²: -0.0354 	RMSE: $116170.91
Epoch: 4 	Loss: 1.3206 	R²: -0.0095 	RMSE: $114704.22
Epoch: 5 	Loss: 1.3105 	R²: -0.0024 	RMSE: $114300.79
Epoch: 6 	Loss: 1.3079 	R²: -0.0007 	RMSE: $114202.32
Epoch: 7 	Loss: 1.3072 	R²: -0.0002 	RMSE: $114177.93
Epoch: 8 	Loss: 1.3068 	R²: -0.0001 	RMSE: $114168.80
Epoch: 9 	Loss: 1.3067 	R²: -0.0000 	RMSE: $114166.57
Epoch: 10 	Loss: 1.3067 	R²: -0.0000 	RMSE: $114166.00
Epoch: 11 	Loss: 1.3067 	R²: -0.0000 	RMSE: $114165.48
Epoch: 12 	Loss: 1.3066 	R²: -0.0000 	RMSE: $114165.25
Epoch: 13 	Loss: 1.3066 	R²: -0.0000 	RMSE: $114165.25
Epoch: 14 	Loss: 1.3067 	R²: -0.0000 	RMSE: $114165.54
Epoch: 15 	Loss: 1.3066 	R²: -0.0000 	RMSE: $114165.21


# Mejor modelo obtenido

Ganó el **modelo 2**, con un Loss de 0.4342, una $R^2$ de 0.6680, y un RMSE de \$65781.69.
Mientras que el modelo 1 obtuvo un Loss de 4686, una $R^2$ de 0.6475, y un RMSE de \$67779.33; y la Red Neuronal 3 obtuvo un Loss de 1.3066, una $R^2$ de ~0, y un RMSE de \$114165.21.

Se descubrió que el optimizador SGD no era apto para esta ocasión. El optimizador Adam obtuvo mejores resultados.

A continuación se evaluará el modelo 2 con valores de prueba.

In [305]:
test_loader = DataLoader(test_dataset, batch_size=10000, shuffle=False)
loss_fn = nn.MSELoss()

model_m2.eval()

# Evaluación con datos de prueba
with torch.no_grad():

    all_preds = []
    all_targets = []
    total_loss = 0.0

    for X_test, y_test in test_loader:
        y_pred = model_m2(X_test)
        loss = loss_fn(y_pred, y_test).item()
        total_loss += loss

        all_preds.append(y_pred)
        all_targets.append(y_test)

    y_pred_all = torch.cat(all_preds).flatten()
    y_true_all = torch.cat(all_targets).flatten()

    # Test R2
    ss_res = torch.sum((y_true_all - y_pred_all) ** 2)
    ss_tot = torch.sum((y_true_all - torch.mean(y_true_all)) ** 2)
    r2 = 1 - ss_res / (ss_tot + 1e-8)

    # Test RMSE
    rmse = torch.sqrt(torch.mean((y_true_all - y_pred_all) ** 2))

    avg_loss = total_loss / len(test_loader)

print(f"Test Loss: {avg_loss:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"RMSE: ${rmse.item() * 100000:.2f}")

Test Loss: 0.4433
R2 Score: 0.6634
RMSE: $66577.82
